# Daily Challenge – Week 7, Day 4: How to Fine-tune LLMs with LoRA (Hugging Face PEFT)
Fine-tune **bigscience/bloomz-560m** on English quotes with LoRA. Run in **Google Colab with a GPU** (Runtime → Change runtime type → GPU); it also runs on CPU, just slowly.

In [ ]:
# The course hint pins peft==0.4.0, which conflicts with current versions of transformers in Colab,
# so we install the current version instead.
%pip install --quiet peft datasets transformers accelerate
!mkdir -p cache

In [ ]:
import os, time
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "bigscience/bloomz-560m"
tokenizer = AutoTokenizer.from_pretrained(model_name)
foundation_model = AutoModelForCausalLM.from_pretrained(model_name)

data = load_dataset("Abirate/english_quotes", split="train[:10%]")      # sample 10%
data = data.map(lambda samples: tokenizer(samples["quote"]), batched=True)
train_sample = data.select(range(5))
display(train_sample)
print(train_sample["quote"])

In [ ]:
# Baseline: what does the model generate BEFORE fine-tuning?
def generate(model, prompt="Two things are infinite: ", max_new_tokens=30):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=max_new_tokens,
        repetition_penalty=1.5,       # discourage repeating the same words
        eos_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.batch_decode(outputs, skip_special_tokens=True)

print("Before fine-tuning:", generate(foundation_model))

In [ ]:
import peft
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=1,                                  # rank of the update matrices
    lora_alpha=1,                         # scaling factor, usually set to 1
    target_modules=["query_key_value"],   # BLOOM's attention projection layer
    lora_dropout=0.05,
    bias="none",                          # don't train the bias parameters
    task_type="CAUSAL_LM",
)

# Add the adapter layers to the foundation model to be trained
peft_model = get_peft_model(foundation_model, lora_config)
print(peft_model.print_trainable_parameters())

Only about **0.01% of the parameters** are trainable. The 560 million original weights stay frozen.

In [ ]:
import transformers
from transformers import TrainingArguments, Trainer

output_directory = os.path.join("./cache/working", "peft_lab_outputs")
training_args = TrainingArguments(
    report_to="none",
    output_dir=output_directory,
    auto_find_batch_size=True,            # find a batch size that fits in memory
    learning_rate=3e-2,                   # higher learning rate than full fine-tuning
    num_train_epochs=2,
    logging_steps=10,
    use_cpu=not torch.cuda.is_available(),
)
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=data.remove_columns(["quote", "author", "tags"]),
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False),
)
trainer.train()

In [ ]:
# Save the fine-tuned LoRA adapter
time_now = time.strftime("%Y%m%d_%H%M%S")
peft_model_path = os.path.join(output_directory, f"peft_model_{time_now}")
trainer.model.save_pretrained(peft_model_path)
print("Saved to:", peft_model_path)
print("Adapter files:", os.listdir(peft_model_path))

The saved adapter is only a few **megabytes**, compared with ~1 GB for the full model. It contains just the LoRA weights.

In [ ]:
# Load the PEFT model using the saved LoRA adapter and a fresh foundation model.
# is_trainable=False avoids further training.
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(model_name)
loaded_model = PeftModel.from_pretrained(base, peft_model_path, is_trainable=False)
loaded_model.to("cuda" if torch.cuda.is_available() else "cpu")

# Generate output tokens
for prompt in ["Two things are infinite: ", "Be yourself; ", "Life is "]:
    print(generate(loaded_model, prompt))

**What to observe:** after fine-tuning, the model's continuations sound more like **short, quotable sayings** (the style of the training data), compared with the base model's output before training. With only 10% of the data, 2 epochs and rank 1, the effect is noticeable but modest: more data, a higher rank (e.g. r = 8) and more epochs would strengthen it.

**Why LoRA / PEFT matters:** we adapted a 560M-parameter model by training only ~0.01% of its weights, which is fast, cheap, possible on a free Colab GPU, and produces a tiny adapter file. The same approach is used to customise multi-billion-parameter LLMs for specific domains, writing styles or tasks.